In [1]:
import platform
import psutil
from typing import Tuple, Union
from timeit import timeit
from warnings import warn

# PyTorch dependencies
import torch
import torch.backends.opt_einsum as opt_einsum
from torch import Tensor

# Internal dependencies
from thoad import config
from thoad import backward, Controller

In [2]:
# control size of tensors
TENSOR_SCALE: Union[int, float] = 1
REPEAT_SCALE: Union[int, float] = 1

In [3]:
sys: platform.uname_result = platform.uname()
print(f"system           {sys.system} {sys.release} {sys.version}")

system           Windows 11 10.0.22631


In [4]:
dev: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if dev.type == 'cuda':
    idx = dev.index if dev.index is not None else 0
    props: "_CudaDeviceProperties" = torch.cuda.get_device_properties(idx)
    name: str = props.name
    total_mem_gb: float = props.total_memory / (1024**3)
    print(f"using device     {dev} -> {name}")
    print(f"device memory    {total_mem_gb:.1f} GB)")
else:
    cpu_name: str = platform.processor() or "CPU"
    print(f"using device     {dev} -> {cpu_name}")
    print(f"physical cores   {psutil.cpu_count(logical=False)}")
    print(f"logical cores    {psutil.cpu_count(logical=True)}")

using device     cuda -> NVIDIA GeForce RTX 4070 Ti
device memory    12.0 GB)


In [5]:
if opt_einsum.is_available():
    opt_einsum.enabled = True
    opt_einsum.strategy = "greedy"
    print("opt_einsum backend enabled")
else:
    warn(
        "opt_einsum backend is not available. "
        "For better performance, install and enable opt_einsum.",
        UserWarning
    )

opt_einsum backend enabled


definition of MLP

In [6]:
def foward_pass(X: Tensor, *params) -> Tensor:
    T: Tensor = X
    for i, P in enumerate(params):
        last_step: bool = i == (len(params) - 1)
        T = T @ P
        T = torch.softmax(T, dim=1) if last_step else torch.relu(T)
    return T.sum()

definition of helper function to meassure differentiation times

In [7]:
def time_differentiation(
        reps: int,
        param_grad: bool,
        keep_batch: bool,
        keep_schwarz: bool,
        order: int,
        X: Tensor,
        *params,
    ) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(param_grad) for P in params]
    def _foward_and_backward() -> None:
        T: Tensor = foward_pass(X, *params)
        ctrl: Controller = backward(
            tensor=T,
            order=order,
            crossings=param_grad,
            keep_batch=keep_batch,
            keep_schwarz=keep_schwarz,
        )
        ctrl.clear()
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

## **Benchmark optimizations**

computational cost w.r.t. **batch size**

In [8]:
for o in [1, 2, 3]:
    print(f"\nORDER {o}")
    for batch_size in [10, 20, 30, 40, 50, 60, 70, 80]:
        param_size: int = int(10 * (1 / o) * TENSOR_SCALE)
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        X: Tensor = torch.rand(size=x_shape, device=dev)
        params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(3)]

        config.SCHWARZ_OPTIMIZATION = False
        reps: int = int(200 * (1/batch_size) * REPEAT_SCALE)
        
        config.BATCH_OPTIMIZATION = False
        regular_time: float = time_differentiation(
            reps, True, True, False, o, X, *params
        )
        config.BATCH_OPTIMIZATION = True
        optimized_time: float = time_differentiation(
            reps, True, True, False, o, X, *params
        )

        print(
            f"batch size: {batch_size:03d} -> "
            f"baseline: {regular_time / reps:.4f}  "
            f"optimized: {optimized_time / reps:.4f}"
        )


ORDER 1
batch size: 010 -> baseline: 0.0151  optimized: 0.0075
batch size: 020 -> baseline: 0.0077  optimized: 0.0076
batch size: 030 -> baseline: 0.0081  optimized: 0.0078
batch size: 040 -> baseline: 0.0077  optimized: 0.0088
batch size: 050 -> baseline: 0.0082  optimized: 0.0082
batch size: 060 -> baseline: 0.0091  optimized: 0.0084
batch size: 070 -> baseline: 0.0082  optimized: 0.0077
batch size: 080 -> baseline: 0.0086  optimized: 0.0092

ORDER 2
batch size: 010 -> baseline: 0.0307  optimized: 0.0321
batch size: 020 -> baseline: 0.0300  optimized: 0.0301
batch size: 030 -> baseline: 0.0305  optimized: 0.0311
batch size: 040 -> baseline: 0.0289  optimized: 0.0313
batch size: 050 -> baseline: 0.0306  optimized: 0.0324
batch size: 060 -> baseline: 0.0278  optimized: 0.0298
batch size: 070 -> baseline: 0.0366  optimized: 0.0350
batch size: 080 -> baseline: 0.0314  optimized: 0.0336

ORDER 3
batch size: 010 -> baseline: 0.1676  optimized: 0.1591
batch size: 020 -> baseline: 0.1724  o

computational cost w.r.t. **depth ~ number of terminal nodes**

In [9]:
for o in [1, 2, 3]:
    print(f"\nORDER {o}")
    for depth in range(2, 7):
        batch_size: int = 10
        param_size: int = int(10 * (1 / o) * TENSOR_SCALE)
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        X: Tensor = torch.rand(size=x_shape, device=dev)
        params: list[Tensor] = [torch.rand(size=p_shape, device=dev) for _ in range(depth)]

        config.BATCH_OPTIMIZATION = False
        reps: int = int(200 * (1/depth) * REPEAT_SCALE)

        config.SCHWARZ_OPTIMIZATION = False
        baseline_time: float = time_differentiation(
            reps, True, False, True, o, X, *params
        )
        config.SCHWARZ_OPTIMIZATION = True
        optimized_time: float = time_differentiation(
            reps, True, False, True, o, X, *params
        )

        print(
            f"graph depth: {depth:02d} -> "
            f"baseline: {baseline_time / reps:.4f}  "
            f"optimized: {optimized_time / reps:.4f}"
        )


ORDER 1
graph depth: 02 -> baseline: 0.0083  optimized: 0.0079
graph depth: 03 -> baseline: 0.0116  optimized: 0.0118
graph depth: 04 -> baseline: 0.0150  optimized: 0.0198
graph depth: 05 -> baseline: 0.0213  optimized: 0.0185
graph depth: 06 -> baseline: 0.0232  optimized: 0.0230

ORDER 2
graph depth: 02 -> baseline: 0.0222  optimized: 0.0154
graph depth: 03 -> baseline: 0.0314  optimized: 0.0260
graph depth: 04 -> baseline: 0.0501  optimized: 0.0379
graph depth: 05 -> baseline: 0.0711  optimized: 0.0535
graph depth: 06 -> baseline: 0.0967  optimized: 0.0726

ORDER 3
graph depth: 02 -> baseline: 0.0734  optimized: 0.0389
graph depth: 03 -> baseline: 0.1722  optimized: 0.0791
graph depth: 04 -> baseline: 0.3544  optimized: 0.1485
graph depth: 05 -> baseline: 0.6638  optimized: 0.2557
graph depth: 06 -> baseline: 1.1661  optimized: 0.4111
